# Steel Production – Regression Model
This notebook trains a model that predicts the `output` value based on 21 input parameters.

In [ ]:
# ── Step 1: Import all required libraries ──────────────────────────────────────
import os                                    # for directory creation
import pandas as pd                          # for loading and handling table data
import numpy as np                           # for numerical operations
import matplotlib.pyplot as plt             # for plotting graphs
from sklearn.ensemble import RandomForestRegressor   # the machine-learning model
from sklearn.model_selection import train_test_split # splits data into train/test
from sklearn.metrics import mean_squared_error, r2_score  # measures model quality

# ── Change working directory to the notebook's own folder ──────────────────────
# VS Code sometimes launches notebooks from a different working directory.
# This line ensures all relative paths are always resolved from src/notebooks/.
os.chdir(os.path.dirname(os.path.abspath("steelproduction.ipynb")))

# Create output directory if it does not exist yet
os.makedirs("../../results/figures", exist_ok=True)

In [ ]:
# ── Step 2: Load the CSV file ───────────────────────────────────────────────────
# pd.read_csv() reads a CSV file and returns a DataFrame (a table-like structure)
DATA_PATH = "../../data/processed/normalized_train_data.csv"
df = pd.read_csv(DATA_PATH)

# Show the first 5 rows so we can inspect the data
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# ── Step 3: Separate input features (X) and output target (y) ──────────────────
# The first column 'output' is what we want to predict (the target)
# All other columns are the input parameters the model learns from
y = df["output"]          # target column – the value we want to predict
X = df.drop("output", axis=1)  # all remaining columns – the input features

print(f"Input features (X): {X.shape[1]} columns")
print(f"Target values  (y): {y.shape[0]} rows")

In [ ]:
# ── Step 4: Split data into training set and test set ──────────────────────────
# We train the model on 80 % of the data and evaluate it on the remaining 20 %.
# random_state=42 ensures the split is the same every time we run this notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

In [ ]:
# ── Step 5: Create and train the model ─────────────────────────────────────────
# RandomForestRegressor builds many decision trees and averages their predictions.
# n_estimators=100  → number of trees in the forest
# random_state=42   → makes results reproducible
model = RandomForestRegressor(n_estimators=100, random_state=42)

# model.fit() is the actual training step –
# the model learns the relationship between X_train and y_train
model.fit(X_train, y_train)

print("Model training complete.")

In [ ]:
# ── Step 6: Make predictions and evaluate model quality ────────────────────────
# model.predict() uses the trained model to predict output values for the test set
y_pred = model.predict(X_test)

# RMSE (Root Mean Squared Error): average prediction error in the same unit as y.
# Lower is better. A value of 0 would mean perfect predictions.
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R² Score: how much of the variance in y is explained by the model.
# 1.0 = perfect, 0.0 = no better than guessing the mean, <0 = worse than mean.
r2 = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.4f}  (average prediction error)")
print(f"R²   : {r2:.4f}  (1.0 = perfect fit)")

In [ ]:
# ── Step 7: Plot – Actual vs. Predicted values ─────────────────────────────────
# Each point represents one sample from the test set.
# x-axis = the true output value from the CSV
# y-axis = the value predicted by our model
# A perfect model would place all points exactly on the diagonal line.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left plot: Scatter plot of actual vs predicted ──────────────────────────
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color="steelblue", label="Samples")

# Draw the ideal diagonal line (perfect prediction)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="red", linewidth=1.5, linestyle="--", label="Perfect fit")

ax.set_xlabel("Actual output (from CSV)")
ax.set_ylabel("Predicted output (model)")
ax.set_title(f"Actual vs. Predicted\nR² = {r2:.4f} | RMSE = {rmse:.4f}")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Right plot: Residuals (prediction errors) ───────────────────────────────
# residual = actual value – predicted value
# A good model has residuals randomly scattered around 0 (no systematic bias).
residuals = y_test.values - y_pred

ax2 = axes[1]
ax2.scatter(y_pred, residuals, alpha=0.3, s=10, color="darkorange")
ax2.axhline(0, color="red", linewidth=1.5, linestyle="--", label="Zero error")
ax2.set_xlabel("Predicted output")
ax2.set_ylabel("Residual (actual – predicted)")
ax2.set_title("Residual Plot")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../../results/figures/actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to results/figures/actual_vs_predicted.png")

In [ ]:
# ── Step 8 (bonus): Feature Importance ─────────────────────────────────────────
# RandomForest can tell us which input columns had the most influence on the output.
# A higher importance value means that feature was more useful for predicting output.
importances = pd.Series(model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
importances_sorted.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature Importance (which inputs matter most)")
ax.set_xlabel("Importance score")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../../results/figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to results/figures/feature_importance.png")